In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

from sentence_transformers import SentenceTransformer

p:\RAG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from pypdf  import PdfReader

"Create README, Create venv using .toml file, add /data to .gitignore". 

In [5]:
# Read PDF and extract text

path_input="data/Code général des impôts-1-373.pdf"

pdf_reader = PdfReader(path_input)
text_content = ""
for page in pdf_reader.pages:
    text_content += page.extract_text()

In [6]:

# Chunk the data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_text(text_content)

print(f"Total pages: {len(pdf_reader.pages)}")
print(f"Total text length: {len(text_content)} characters")
print(f"Number of chunks: {len(chunks)}\n")

print("First chunk preview:\n\n"+f"{chunks[0]}\n")
print("Chunk No 10 preview:\n\n"+f"{chunks[9]}\n")
print("Chunk No 100 preview:\n\n"+f"{chunks[99]}\n")

Total pages: 373
Total text length: 1803712 characters
Number of chunks: 2220

First chunk preview:

Code général des impôts
Dernière modification: 2026-03-14
Edition : 2026-03-14
2419 articles avec 5773 liens
818 références externes
Ce code ne contient que du droit positif français,
les articles et éléments abrogés ne sont pas inclus.
Il est recalculé au fur et à mesure des mises à jour.
Pensez à actualiser votre copie régulièrement à partir de codes.droit.org.
Ces codes ont pour objectif de démontrer l’utilité de l’ouverture des données publiques juridiques tant législatives que
jurisprudentielles. Il s’y ajoute une promotion du mouvement Open Science Juridique avec une incitation au dépôt du
texte intégral en accès ouvert des articles de doctrine venant du monde professionnel (Grande Bibliothèque du Droit) et
universitaire (HAL-CNRS).
Traitements effectués à partir des données issues des APIs Legifrance et Judilibre. droit.org remercie les acteurs du
Web qui autorisent des liens ver

In [7]:

# Chunk the data using Markdown headers as delimiters 

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Livre"),
        ("##", "Partie"),
        ("###", "Titre"),
        ("####", "Chapitre")
    ]
)

chunks_avec_meta = splitter.split_text(text_content)


print(chunks_avec_meta[0].metadata)

{}


In [8]:


# Chargement du modèle (téléchargé automatiquement la première fois)
modele = SentenceTransformer('all-MiniLM-L6-v2')

# Vectorisation de tous les chunks d'un coup
vecteurs = modele.encode(chunks, show_progress_bar=True)

print(f"Nombre de chunks vectorisés : {len(vecteurs)}")
print(f"Dimensions de chaque vecteur : {vecteurs.shape[1]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Nombre de chunks vectorisés : 2220
Dimensions de chaque vecteur : 384


In [9]:
print(type(vecteurs))
print(len(vecteurs))
print(type(vecteurs[0]))
print(len(vecteurs[0]))

<class 'numpy.ndarray'>
2220
<class 'numpy.ndarray'>
384


In [10]:
import chromadb

# Connexion à ChromaDB en mode persistant (sauvegarde sur disque)
client = chromadb.PersistentClient(path="./ma_bdd_juridique")

# Création de la collection (comme une table en SQL)
collection = client.get_or_create_collection(
    name="code_civil",
    metadata={"hnsw:space": "cosine"}  # on utilise la similarité cosinus
)

In [11]:
# Stockage des chunks + vecteurs + métadonnées
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=vecteurs.tolist(),
    metadatas=[
        {
            "source": "Code civil",
            "chunk_index": i
        }
        for i in range(len(chunks))
    ]
)

print(f"✅ {collection.count()} chunks stockés dans ChromaDB")

✅ 2220 chunks stockés dans ChromaDB


In [12]:
resultats = collection.get(
    ids=["chunk_0", "chunk_1"],
    include=["documents", "metadatas"]
)

for doc, meta in zip(resultats["documents"], resultats["metadatas"]):
    print(f"Source : {meta['source']}")
    print(f"Texte  : {doc[:100]}...")
    print("---")

Source : Code civil
Texte  : Code général des impôts
Dernière modification: 2026-03-14
Edition : 2026-03-14
2419 articles avec 57...
---
Source : Code civil
Texte  : Web qui autorisent des liens vers leur production : Dictionnaire du Droit Privé  (réalisé par MM. Se...
---


In [14]:
# from retriever import rechercher, formater_contexte

# chunks = rechercher("Quels sont les droits du locataire ?")
# print(formater_contexte(chunks))

In [16]:
# ─────────────────────────────────────────────
# Connexion à la BDD existante (lecture seule)
# ─────────────────────────────────────────────
client = chromadb.PersistentClient(path="./ma_bdd_juridique")
collection = client.get_collection(name="code_civil")

# Même modèle que celui utilisé pour créer la BDD
modele = SentenceTransformer('all-MiniLM-L6-v2')


def rechercher(question: str, k: int = 4) -> list[dict]:
    """
    Vectorise la question et retourne les k chunks les plus pertinents.

    Retourne une liste de dicts :
    [
        {"texte": "...", "source": "Code civil", "chunk_index": 42, "score": 0.87},
        ...
    ]
    """
    # 1. Vectoriser la question avec le même modèle
    vecteur_question = modele.encode(question).tolist()

    # 2. Recherche par similarité cosinus dans ChromaDB
    resultats = collection.query(
        query_embeddings=[vecteur_question],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    # 3. Formater les résultats
    chunks_pertinents = []
    for i in range(len(resultats["documents"][0])):
        chunks_pertinents.append({
            "texte": resultats["documents"][0][i],
            "source": resultats["metadatas"][0][i].get("source", "inconnu"),
            "chunk_index": resultats["metadatas"][0][i].get("chunk_index", i),
            "score": round(1 - resultats["distances"][0][i], 3)  # distance → similarité
        })

    return chunks_pertinents


def formater_contexte(chunks: list[dict]) -> str:
    """
    Formate les chunks en contexte lisible pour le prompt Mistral.
    """
    contexte = ""
    for i, chunk in enumerate(chunks, 1):
        contexte += f"[Extrait {i} — {chunk['source']} — pertinence : {chunk['score']}]\n"
        contexte += chunk["texte"] + "\n\n"
    return contexte.strip()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
chunks = rechercher("Quels sont les droits du locataire ?")
chunks

[{'texte': "le revenu et qui sont considérés comme des éléments d'actif affectés à l'exercice de la profession au sens du\nI de l'article 151 nonies ;\n3° Le cédant cesse toute fonction dans l'entreprise individuelle cédée ou dans la société ou le groupement dont\nles droits ou parts sont cédés et fait valoir ses droits à la retraite, dans les deux années suivant ou précédant\nla cession ;\n4° Le cédant ne doit pas détenir, directement ou indirectement, plus de 50 % des droits de vote ou des droits\ndans les bénéfices sociaux de l'entreprise cessionnaire ;\n5° L'entreprise individuelle cédée ou la société ou le groupement dont les droits ou parts sont cédés emploie\nmoins de deux cent cinquante salariés et soit a réalisé un chiffre d'affaires annuel inférieur à 50 millions d'euros\nau cours de l'exercice, soit a un total de bilan inférieur à 43 millions d'euros ;\n6° Le capital ou les droits de vote de la société ou du groupement dont les droits ou parts sont cédés ne",
  'source': 'Co

In [19]:
print(formater_contexte(chunks))

[Extrait 1 — Code civil — pertinence : 0.581]
le revenu et qui sont considérés comme des éléments d'actif affectés à l'exercice de la profession au sens du
I de l'article 151 nonies ;
3° Le cédant cesse toute fonction dans l'entreprise individuelle cédée ou dans la société ou le groupement dont
les droits ou parts sont cédés et fait valoir ses droits à la retraite, dans les deux années suivant ou précédant
la cession ;
4° Le cédant ne doit pas détenir, directement ou indirectement, plus de 50 % des droits de vote ou des droits
dans les bénéfices sociaux de l'entreprise cessionnaire ;
5° L'entreprise individuelle cédée ou la société ou le groupement dont les droits ou parts sont cédés emploie
moins de deux cent cinquante salariés et soit a réalisé un chiffre d'affaires annuel inférieur à 50 millions d'euros
au cours de l'exercice, soit a un total de bilan inférieur à 43 millions d'euros ;
6° Le capital ou les droits de vote de la société ou du groupement dont les droits ou parts sont cé